# AXI test 14: does snapping a hidden vector to the nearest allowed token in a hyperbolic ball do anything that masking does not? (Qwen2.5-1.5B-Instruct, free Colab T4)
One test, from section 3 of the AI-written roadmap the owner pasted on 17 Oct 2026 (I do not know whether any of it is Rajnish Choubey's). A small open model reads a service graph and a request and then the **correct reply, one token at a time** (the reply is fed in, nothing is generated). At every token the program stores the model's last hidden vector, the token the model itself would pick, and the tokens the repository's grammar allows. Then, on the CPU, nine ways of choosing the token are scored against the correct one: the model's own pick, plain masking, nearest-token rules in a flat and in a hyperbolic (Poincare) space, and three small trained heads.

* **13 reading rules**, each with my probability, were written and locked BEFORE this run in `docs/HYPERBOLIC_SNAP_TEST.md`. My chance for the roadmap's claim in its fair form is **8 %**. One thing is true before the run for any rule that only picks among allowed tokens: it can never write a token the grammar refuses (the roadmap's "0.00%"). That is by construction, not a finding.
* Two conditions: `full` (the repository's prompt, which describes the syntax) and `bare` (no description). 400 tasks each.

**What to know**
* **Cost: $0.** A local open model, no key. It needs the **free T4 GPU**: Runtime menu -> *Change runtime type* -> **T4 GPU** -> Save.
* **Time (my estimate, not measured):** download of the model about 3 GB, a few minutes; collecting both conditions about 10 to 20 minutes; the analysis on the CPU a few minutes. If Colab disconnects: reconnect, run Cells 2 to 5, then Cell 8 again (finished parts are saved to your Drive and skipped).
* **Cell 6 and Cell 7 are safety checks** (they stop if the replies do not fit the grammar, or if the stored vector is not the one that goes into the output head). If either says STOP, send the printout.
* **Do not paste any key or password anywhere.** This notebook needs none.

Run each cell **one at a time, top to bottom**, wait for the green tick before the next one.

### Cell 1: check the GPU (it must say Tesla T4)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

### Cell 2: upload the code. A button 'Choose files' appears: pick `axi-colab-tests.zip` (the one that comes with this notebook)

In [ ]:
from google.colab import files
up = files.upload()
name = list(up)[0]
!rm -rf /content/axi && mkdir -p /content/axi && unzip -q -o "{name}" -d /content/axi
!ls /content/axi/python

### Cell 3: connect Google Drive (a pop-up asks permission: tap Connect / Allow). This is where your progress is kept

In [ ]:
import sys, os, json, time
sys.path.insert(0, "/content/axi/python")
from colab.colab_persist import open_store
store = open_store("/content/out", "axi_hypsnap")

### Cell 4: load the model (about 3 GB, a few minutes) and make the 400 tasks

In [ ]:
import numpy as np, torch
from google.colab import files
import axi.experiments.hypsnap as HS
from colab.colab_llm_experiment import build_hf_llm
llm, model = build_hf_llm(HS.MODEL, "cuda", load_4bit=False)
tok, table, eos = llm.tok, llm.table, llm.eos
runner, prompt_ids_fn, unembed = HS.torch_runner(model, tok)
encode = lambda s: tok.encode(s, add_special_tokens=False)
tasks = HS.make_tasks(HS.N_TASKS, HS.TASK_SEED)
print("model:", HS.MODEL, "| hidden size:", model.config.hidden_size, "| tasks:", len(tasks), "| GPU memory used (GB):", round(torch.cuda.memory_allocated() / 1e9, 1))

### Cell 5: first safety check (no model time, a few seconds). Every correct reply, cut into tokens the way this tokenizer cuts it, must be allowed by the grammar from start to end. If one is not, the pre-registered line 2 is false and the test says so; it still helps to know now

In [ ]:
bad = HS.check_walkable(table, encode, eos, tasks)
print("replies the grammar refuses somewhere:", len(bad), "of", len(tasks))
for i, j, g in bad[:8]: print("  task", i, "position", j, "token", g, repr(tok.decode([g])), "| reply:", tasks[i].ideal[:70])
if bad: print("\nSTOP is NOT required: the run can go on (the page says what a refused token means). Send this printout with your result.")
else: print("\nALL REPLIES ARE ALLOWED. Go on to Cell 6.")

### Cell 6: second safety check (about 1 minute of model time): 5 tasks. Checks the shapes, that the stored `h_pre` followed by the model's final norm gives the stored `h_post` (so the hook read the right tensor), and shows what the model picks against what the grammar allows

In [ ]:
d5 = HS.collect(runner, prompt_ids_fn, encode, table, eos, tasks[:5], "full")
print("positions:", len(d5["ex"]), "| broken replies:", d5["broken"], "| h_post shape:", d5["h_post"].shape)
def apply_norm(x):
    with torch.no_grad(): return model.model.norm(torch.tensor(x, device="cuda").half()).float().cpu().numpy()
err = HS.norm_identity_error(d5, apply_norm); print("final norm of h_pre against stored h_post, largest difference as a share of the largest value:", round(err, 5))
assert d5["h_post"].shape[1] == model.config.hidden_size and d5["h_pre"].shape == d5["h_post"].shape, "STOP: the stored vectors have the wrong shape. Send the printout."
assert err < 0.02, "STOP: the stored vector is not the one that goes into the output head. Send the printout."
for pos in range(min(14, len(d5["ex"]))):
    n_allowed = int(d5["a_ptr"][pos + 1] - d5["a_ptr"][pos]); ft = int(d5["free_top"][pos]); ok = bool((d5["a_ids"][d5["a_ptr"][pos]:d5["a_ptr"][pos + 1]] == ft).any())
    print(f"  reply token {int(d5['gold'][pos]):6d} {tok.decode([int(d5['gold'][pos])])!r:14}  model picks {ft:6d} {tok.decode([ft])!r:14} allowed by the grammar: {ok!s:5}  ({n_allowed} tokens allowed here)")
print("\nALL CHECKS OK. Go on to Cell 7.")

### Cell 7: THE COLLECTION. Both conditions, 400 replies each (the counter shows the replies done; each finished condition is saved to your Drive). About 10 to 20 minutes

In [ ]:
T0 = time.time()
def say(cond, i, n, pos): print(f"  {cond}: {i}/{n} replies, {pos} positions   {time.time() - T0:5.0f} s", end="\r")
raw = {}
for cond in HS.CONDITIONS:
    fn = f"axi_hs_{cond}_raw.npz"; print(f"=== {cond}: {store.restore(fn)}")
    if os.path.exists(store.path(fn)): raw[cond] = HS.load_data(store.path(fn)); print("   loaded from Drive:", len(raw[cond]["ex"]), "positions"); continue
    raw[cond] = HS.collect(runner, prompt_ids_fn, encode, table, eos, tasks, cond, say=say)
    HS.save_data(store.path(fn), raw[cond]); store.persist(fn); print(); print("   done:", len(raw[cond]["ex"]), "positions, broken replies:", raw[cond]["broken"])
HS.attach_unembedding([raw["full"], raw["bare"]], unembed)
for cond in HS.CONDITIONS:
    fn = f"axi_hs_{cond}.npz"; HS.save_data(store.path(fn), raw[cond]); store.persist(fn)
print("collection finished in", round((time.time() - T0) / 60, 1), "minutes | tokens allowed somewhere:", len(raw["full"]["U_ids"]))

### Cell 8: THE ANALYSIS (on the CPU; a few minutes): the nine ways of choosing, both conditions, and the 13 reading rules. Then your file downloads

In [ ]:
import subprocess
out = store.path("axi_hypsnap.json")
r = subprocess.run([sys.executable, "-u", "-m", "axi.experiments.hypsnap", "--full", store.path("axi_hs_full.npz"), "--bare", store.path("axi_hs_bare.npz"), "--out", out], cwd="/content/axi/python", env=dict(os.environ, PYTHONPATH="/content/axi/python"), capture_output=True, text=True)
print(r.stdout[-6000:]); print(r.stderr[-1500:]); assert r.returncode == 0, "STOP: the analysis failed. Send the printout."
store.persist("axi_hypsnap.json"); files.download(out)

### Done
File downloaded: `axi_hypsnap.json`. Upload it to the chat, with a screenshot of the two printed tables and of the 13 reading-rule lines. The two `.npz` files with the stored vectors stay on your Drive (folder `axi_hypsnap`); I do not need them unless something looks wrong. Do not paste any keys anywhere.

When the numbers are in, `docs/HYPERBOLIC_SNAP_TEST.md` gets a "Result" section **below** the locked predictions; nothing above it changes.